In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F   
from torch.utils.data import Dataset, DataLoader

In [ ]:
SR = 22050
CLIP_SECONDS = 5
SAMPLES = SR * CLIP_SECONDS

N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 256

In [ ]:
BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
STEMS_PATH = f"{BASE_PATH}/genres_stems"
NOISE_PATH = f"{BASE_PATH}/ESC-50-master/audio"
TEST_PATH = f"{BASE_PATH}/mashups"
SUB_PATH = f"{BASE_PATH}/sample_submission.csv"

In [ ]:
genres = sorted(os.listdir(STEMS_PATH))
label_to_idx = {g: i for i, g in enumerate(genres)}
idx_to_label = {i: g for g, i in label_to_idx.items()}

print("Genres:", genres)
print("Total genres:", len(genres))

In [ ]:
noise_files = [
    os.path.join(NOISE_PATH, f)
    for f in os.listdir(NOISE_PATH)
    if f.endswith(".wav")
]

print("Noise files:", len(noise_files))

In [ ]:
class MashupDataset(Dataset):
    def __init__(self, stems_path, noise_files=None):
        self.samples = []
        self.noise_files = noise_files

        print("Scanning dataset...")

        for genre in genres:
            genre_path = os.path.join(stems_path, genre)

            for song in os.listdir(genre_path):
                song_path = os.path.join(genre_path, song)

                stems = {
                    "drums": os.path.join(song_path, "drums.wav"),
                    "bass": os.path.join(song_path, "bass.wav"),
                    "vocals": os.path.join(song_path, "vocals.wav"),
                    "other": os.path.join(song_path, "other.wav"),
                }

                # include song if ANY stem exists
                if any(os.path.exists(s) for s in stems.values()):
                    self.samples.append((stems, label_to_idx[genre]))

        print("Total training songs:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def _crop(self, x):
        if len(x) > SAMPLES:
            start = np.random.randint(0, len(x)-SAMPLES)
            return x[start:start+SAMPLES]
        return np.pad(x,(0,SAMPLES-len(x)))

    def __getitem__(self, idx):
        stems, label = self.samples[idx]

        mix = np.zeros(SAMPLES, dtype=np.float32)

        # 🔥 random stem mix
        for stem_name in ["drums","bass","vocals","other"]:
            path = stems[stem_name]
            if os.path.exists(path) and np.random.rand() < 0.9:
                y,_ = librosa.load(path, sr=SR, mono=True)
                y = self._crop(y)
                gain = np.random.uniform(0.6,1.3)
                mix += gain * y

        # 🔥 add noise
        if self.noise_files and np.random.rand() < 0.5:
            noise_path = random.choice(self.noise_files)
            noise,_ = librosa.load(noise_path, sr=SR, mono=True)
            noise = self._crop(noise)
            mix += 0.2 * noise

        # normalize
        mix = mix / (np.max(np.abs(mix)) + 1e-6)

        mel = librosa.feature.melspectrogram(
            y=mix,
            sr=SR,
            n_mels=N_MELS,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH
        )

        mel = librosa.power_to_db(mel, ref=np.max)
        mel = (mel - mel.mean())/(mel.std()+1e-6)

        mel = torch.tensor(mel).unsqueeze(0).float()
        label = torch.tensor(label).long()

        return mel, label


In [ ]:
dataset = MashupDataset(STEMS_PATH, noise_files)
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)


In [ ]:
class AudioCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self._to_linear = None
        self._infer()

        self.fc = nn.Sequential(
            nn.Linear(self._to_linear, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def _infer(self):
        with torch.no_grad():
            x = torch.randn(1,1,N_MELS,int(np.ceil(SAMPLES/HOP_LENGTH)))
            x = self.conv(x)
            self._to_linear = x.view(1,-1).shape[1]

    def forward(self,x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:

model = AudioCNN(num_classes=len(genres)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)

epochs = 36

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss / len(loader):.4f}")


In [ ]:
class TestDataset(Dataset):
    def __init__(self, folder):
        self.files = sorted([
            os.path.join(folder, f)
            for f in os.listdir(folder)
            if f.endswith(".wav")
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]
        y, _ = librosa.load(file, sr=SR, mono=True)

        if len(y) > SAMPLES:
            y = y[:SAMPLES]
        else:
            y = np.pad(y, (0, SAMPLES - len(y)))

        mel = librosa.feature.melspectrogram(
            y=y,
            sr=SR,
            n_mels=N_MELS,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH
        )

        mel = librosa.power_to_db(mel, ref=np.max)
        mel = (mel - mel.mean())/(mel.std()+1e-6)

        mel = torch.tensor(mel).unsqueeze(0).float()
        return mel, os.path.basename(file)


In [ ]:
test_loader = DataLoader(TestDataset(TEST_PATH), batch_size=16)
model.eval()

preds, names = [], []

with torch.no_grad():
    for x, n in test_loader:
        x = x.to(device)
        p = torch.argmax(model(x), dim=1).cpu().numpy()
        preds.extend(p)
        names.extend(n)

genre_preds = [idx_to_label[p] for p in preds]

submission = pd.read_csv(SUB_PATH)
submission["genre"] = genre_preds
submission.to_csv("submission.csv", index=False)

print("✅ submission.csv saved")
submission.head()

In [ ]:
submission.shape